# MCP Toolbox in Docker + OpenRouter tool calls (demo)

This notebook explains how the tutorial stack wires **Google MCP Toolbox** into Docker with YAML tool definitions, then runs a **minimal** chat loop (OpenRouter + `toolbox-core`) here in plain cells for learning.

**Prerequisites:** from the repo root, `docker compose up --build` (Postgres, Mongo, seed, toolbox), a venv with `pip install -e .`, and `OPENROUTER_API_KEY` in `.env` or the environment. Default Toolbox URL: `http://127.0.0.1:5050` (host port maps to port 5000 in the container).

**Official docs:** [MCP Toolbox for Databases](https://mcp-toolbox.dev/documentation/introduction/) (architecture and features)  |  [Toolbox YAML configuration](https://mcp-toolbox.dev/documentation/configure.md)  |  [MCP quickstart](https://mcp-toolbox.dev/documentation/getting-started/mcp_quickstart/)  |  [Python `toolbox-core` SDK](https://googleapis.github.io/genai-toolbox/sdks/python-sdk/core/) ([sync usage](https://googleapis.github.io/genai-toolbox/sdks/python-sdk/core/#synchronous-usage))  |  [OpenRouter API](https://openrouter.ai/docs/quickstart)  |  [OpenAI function calling (tools)](https://platform.openai.com/docs/guides/function-calling)  |  [OpenAI Python library](https://github.com/openai/openai-python).

Run cells in order.


## 1. Docker Compose and YAML tools

**Before this section (and the rest of the notebook):** follow the repo [README](../README.md) so **Docker Compose is already running** (README **step 1**, and optionally **step 2** to sanity-check the DBs) and your **Python env is set up** (README **step 3**: venv activated, `pip install -e .`, and a Jupyter kernel pointed at that venv if you use one). Section 1 is read-only about YAML, but later cells call Toolbox on `TOOLBOX_BASE_URL` and need the editable Python install from the README; without the stack and install, those steps will fail.

At the **repo root**, `docker-compose.yml` defines `postgres`, `mongo`, a one-shot `seed` job, and **`toolbox`**. The Toolbox container:

- Mounts **`toolbox/config`** read-only at **`/config`** inside the container.
- Starts with **`--config-folder /config/tools`**, so every `*.yaml` in **`toolbox/config/tools/`** is merged (file order: `01_`, `02_`, …).
- Gets **database hostnames** as Docker service names (`postgres`, `mongo`), not `localhost`, because Toolbox runs on the compose network.
- Publishes **host port 5050 → container port 5000**; Python on the host uses **`TOOLBOX_BASE_URL`** (default `http://127.0.0.1:5050`).

YAML files declare **`kind: tool`** (SQL or Mongo operations with parameters) and **`kind: toolset`** (named lists of tools). This tutorial loads toolset **`combined`** (Postgres + Mongo)—see `toolbox/config/tools/04_toolsets.yaml`.


**Toolbox docs:** overview in the [introduction](https://mcp-toolbox.dev/documentation/introduction/); how services and tools are declared in [configuration](https://mcp-toolbox.dev/documentation/configure.md); using Toolbox as an MCP server in the [MCP quickstart](https://mcp-toolbox.dev/documentation/getting-started/mcp_quickstart/).

The **next two cells** show minimal **Toolbox YAML** from this repo: one `kind: source` (database connection) and one `kind: tool` (a small Postgres SQL tool the model can call).


### 1.1 Example YAML: a **source**

Toolbox merges every `*.yaml` under `toolbox/config/tools/` (see section 1 above). A **source** is a named database connection. Tools point at it with `source: <name>`. Below is the Postgres source from [`toolbox/config/tools/01_sources.yaml`](../toolbox/config/tools/01_sources.yaml) (first document in that file). `${...}` values come from the Toolbox container environment.


**Docs:** how `kind: source` fits the Toolbox config model: [configuration](https://mcp-toolbox.dev/documentation/configure.md).

```yaml
kind: source
name: olympics-postgres
type: postgres
host: ${POSTGRES_HOST}
port: ${POSTGRES_PORT}
database: ${POSTGRES_DB}
user: ${POSTGRES_USER}
password: ${POSTGRES_PASSWORD}
```


### 1.2 Example YAML: a **tool**

A **tool** is a separate YAML document with `kind: tool`: a stable `name`, a `type` (here `postgres-sql` runs `statement` on the linked `source`), parameters the model will fill in (mapped to `$1`, `$2`, ... in order), and optional `annotations` hints. This example is `pg_search_countries` from [`toolbox/config/tools/02_postgres_tools.yaml`](../toolbox/config/tools/02_postgres_tools.yaml).


**Docs:** tool documents (`kind: tool`, `statement`, parameters) are covered under [configuration](https://mcp-toolbox.dev/documentation/configure.md).

```yaml
kind: tool
name: pg_search_countries
type: postgres-sql
source: olympics-postgres
description: >
  Search countries by name fragment and return NOC codes (National Olympic Committee).
  Use before medal tools when the user says e.g. "United States" instead of "USA".
parameters:
  - name: name_fragment
    type: string
    description: Substring to match against country name (case-insensitive).
statement: |
  SELECT noc, country
  FROM countries
  WHERE country ILIKE '%' || $1 || '%'
  ORDER BY country ASC
  LIMIT 50
annotations:
  readOnlyHint: true
  destructiveHint: false
```


## 2. Environment and settings

Load the repo `.env` and build `Settings` (OpenRouter + Toolbox URLs, model id). Environment variable names match [`.env.example`](../.env.example).

**Docs:** [OpenRouter quickstart](https://openrouter.ai/docs/quickstart) (keys and OpenAI-compatible base URL); [`toolbox-core` on PyPI](https://pypi.org/project/toolbox-core/); [Python SDK core guide](https://googleapis.github.io/genai-toolbox/sdks/python-sdk/core/) ([sync client](https://googleapis.github.io/genai-toolbox/sdks/python-sdk/core/#synchronous-usage)).


In [ ]:
import os
from dataclasses import dataclass
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
from toolbox_core import ToolboxSyncClient
from toolbox_core.protocol import Protocol
from toolbox_core.sync_tool import ToolboxSyncTool


AGENT_MAX_TOOL_ROUNDS = 16


@dataclass(frozen=True)
class Settings:
    """URLs and keys from the environment; ``agent_max_tool_rounds`` is fixed here."""

    openrouter_api_key: str
    openrouter_base_url: str
    openrouter_model: str
    toolbox_base_url: str
    agent_max_tool_rounds: int

    @staticmethod
    def from_env() -> "Settings":
        key = (os.environ.get("OPENROUTER_API_KEY") or "").strip()
        base = (
            os.environ.get("OPENROUTER_BASE_URL") or "https://openrouter.ai/api/v1"
        ).strip()
        model = (os.environ.get("OPENROUTER_MODEL") or "openai/gpt-4o-mini").strip()
        toolbox = (
            (os.environ.get("TOOLBOX_BASE_URL") or "http://127.0.0.1:5050")
            .strip()
            .rstrip("/")
        )
        return Settings(
            openrouter_api_key=key,
            openrouter_base_url=base,
            openrouter_model=model,
            toolbox_base_url=toolbox,
            agent_max_tool_rounds=AGENT_MAX_TOOL_ROUNDS,
        )


def repo_root() -> Path:
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / "pyproject.toml").is_file():
            return d
    raise RuntimeError("Run from inside the repo clone (need pyproject.toml).")


load_dotenv(repo_root() / ".env")
settings = Settings.from_env()


## 3. Tool-calling loop (happy path)

The cells below build a **minimal tool-calling loop** in **small steps**: types -> OpenAI tool list -> Toolbox session -> one completion at a time -> full loop.

**Docs:** [Python SDK: synchronous usage](https://googleapis.github.io/genai-toolbox/sdks/python-sdk/core/#synchronous-usage), [loading tools / toolsets](https://googleapis.github.io/genai-toolbox/sdks/python-sdk/core/#loading-tools); OpenAI-style tool calls in [function calling](https://platform.openai.com/docs/guides/function-calling); [OpenRouter](https://openrouter.ai/docs/quickstart).


### 3.1 Imports and JSON Schema type names

OpenAI-style tool definitions use JSON Schema inside `function.parameters`. Toolbox parameter types are short strings; we map them to schema types the API accepts.

**Docs:** [OpenAI function calling / tool schema](https://platform.openai.com/docs/guides/function-calling); [OpenAI Python SDK](https://github.com/openai/openai-python).


In [ ]:
import json
from collections.abc import Sequence
from typing import Any

# ``json`` parses each tool's argument string from the model into a Python dict.
# ``Sequence`` / ``Any`` keep type hints readable for ``ToolboxSyncTool`` lists and message dicts.

# Toolbox names parameter kinds with short strings; OpenAI expects JSON Schema types here.
_JSON_TYPES = {
    "string": "string",
    "integer": "integer",
    "float": "number",
    "boolean": "boolean",
    "array": "array",
    "object": "object",
}


### 3.2 Toolbox tools -> OpenAI ``tools`` list

Each loaded `ToolboxSyncTool` becomes one `{type: function, function: {...}}` entry for `chat.completions.create(..., tools=...)`.

**Docs:** [Python SDK: loading tools](https://googleapis.github.io/genai-toolbox/sdks/python-sdk/core/#loading-tools); [OpenAI function calling](https://platform.openai.com/docs/guides/function-calling).


In [ ]:
def toolbox_tools_to_openai(tools: Sequence[ToolboxSyncTool]) -> list[dict[str, Any]]:
    """Map toolbox_core tools to OpenAI chat ``tools=`` shape (happy path)."""
    # One dict per tool; the chat API exposes these as functions the model may invoke.
    out: list[dict[str, Any]] = []
    for t in tools:
        # JSON Schema object for this function's arguments (names, types, which are required).
        props: dict[str, Any] = {}
        required: list[str] = []
        # ``t._params`` is toolbox_core's internal list of parameter metadata for this tool.
        for p in t._params:
            props[p.name] = {
                "type": _JSON_TYPES.get(p.type, "string"),
                "description": p.description,
            }
            if p.required:
                required.append(p.name)
        out.append(
            {
                "type": "function",
                "function": {
                    # Must match the strings the model emits in ``tool_calls[].function.name``.
                    "name": t._name,
                    # Shown to the model only; helps it choose the right tool and arguments.
                    "description": (t._description or "").strip(),
                    "parameters": {
                        "type": "object",
                        "properties": props,
                        "required": required,
                        "additionalProperties": False,
                    },
                },
            }
        )
    return out


### 3.3 Look up tools by function name

When the model returns `tool_calls`, each call names a function. We index the live `ToolboxSyncTool` objects by `t._name` so we can run `tool(**args)`.

**Docs:** [Python SDK: invoking tools](https://googleapis.github.io/genai-toolbox/sdks/python-sdk/core/#invoking-tools).


In [ ]:
def tool_by_name(tools: Sequence[ToolboxSyncTool]) -> dict[str, ToolboxSyncTool]:
    # When the model picks ``function.name``, we look up the live callable bound to Toolbox.
    return {t._name: t for t in tools}


### 3.4 OpenRouter via the OpenAI SDK

Same client library as OpenAI; `base_url` and `api_key` point at OpenRouter. Optional headers identify this repo to OpenRouter.

**Docs:** [OpenRouter quickstart](https://openrouter.ai/docs/quickstart); [OpenAI Python library](https://github.com/openai/openai-python).


In [ ]:
def openrouter_client(settings: Settings) -> OpenAI:
    # Same ``openai`` SDK as OpenAI; ``base_url`` + ``api_key`` point at OpenRouter instead.
    return OpenAI(
        api_key=settings.openrouter_api_key,
        base_url=settings.openrouter_base_url,
        default_headers={
            "HTTP-Referer": "https://github.com/Rotational-io/mcp-tutorial",
            "X-Title": "mcp-tutorial",
        },
    )


### 3.5 Build the first `messages` list

Standard chat roles: `system`, then `user`. We append assistant and `tool` rows later when the model uses tools.

**Docs:** [OpenAI function calling (message format)](https://platform.openai.com/docs/guides/function-calling).


In [ ]:
def initial_messages(
    system_prompt: str,
    user_prompt: str,
    cap: int,
) -> list[dict[str, Any]]:
    # Keep the model's plan aligned with how many tool-bearing turns the client allows.
    sys_text = (
        system_prompt.rstrip()
        + f"\n\nAt most {cap} model turns may include tool calls."
    )
    # Chat Completions expects a time-ordered list; we will append assistant + tool rows next.
    return [
        {"role": "system", "content": sys_text},
        {"role": "user", "content": user_prompt},
    ]


### 3.6 Ask the model once

Pass the **whole** transcript so far plus the static `tools` list. `tool_choice="auto"` lets the model decide whether to call tools.

**Docs:** [OpenAI function calling](https://platform.openai.com/docs/guides/function-calling); [OpenRouter](https://openrouter.ai/docs/quickstart).


In [ ]:
def call_model(
    oai: OpenAI,
    settings: Settings,
    messages: list[dict[str, Any]],
    oai_tools: list[dict[str, Any]],
):
    """One ``chat.completions`` round (reply may include ``tool_calls``)."""
    return oai.chat.completions.create(
        model=settings.openrouter_model,
        messages=messages,  # full transcript so far (system, user, prior tools, ...)
        tools=oai_tools,  # same catalog every round for this Toolbox session
        tool_choice="auto",  # model may answer with text only, call tools, or mix
    )


### 3.7 Record tool calls and append tool results

If the assistant returns `tool_calls`, the API format requires:

1. An `assistant` message that includes the same `id`s for each call.
2. One `tool` message per call, with matching `tool_call_id` and string `content` (here: Toolbox output).

**Docs:** [OpenAI function calling (responding with tool outputs)](https://platform.openai.com/docs/guides/function-calling).


In [ ]:
def append_assistant_with_tool_calls(messages: list[dict[str, Any]], msg) -> None:
    """Append assistant turn including ``tool_calls`` (ids link to the next rows)."""
    # Without this assistant row (including ids), the API rejects the following ``tool`` rows.
    messages.append(
        {
            "role": "assistant",
            "content": msg.content,  # often empty when the model only wants tools
            "tool_calls": [
                {
                    "id": tc.id,  # must match ``tool_call_id`` on each tool message below
                    "type": "function",
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments,  # JSON object as a string
                    },
                }
                for tc in msg.tool_calls
            ],
        }
    )


def append_tool_outputs(
    messages: list[dict[str, Any]],
    msg,
    by_name: dict[str, ToolboxSyncTool],
) -> None:
    """Parse JSON arguments, invoke Toolbox tool, append ``role: tool`` rows."""
    for tc in msg.tool_calls:
        args = json.loads(tc.function.arguments or "{}")
        # Blocking call into MCP Toolbox; ``payload`` is a string (often JSON) for the model.
        payload = by_name[tc.function.name](**args)
        messages.append(
            {
                "role": "tool",
                "tool_call_id": tc.id,
                "content": payload,
            }
        )


### 3.8 Put it together: `chat_with_tools`

Open one MCP session to Toolbox, load the `combined` toolset, then loop: `call_model` -> if there are tool calls, append rows and repeat -> otherwise return the assistant text.

**Docs:** [Python SDK: synchronous usage](https://googleapis.github.io/genai-toolbox/sdks/python-sdk/core/#synchronous-usage); [Toolbox configuration](https://mcp-toolbox.dev/documentation/configure.md) (toolset names come from YAML); [OpenAI function calling](https://platform.openai.com/docs/guides/function-calling).


In [ ]:
def chat_with_tools(
    settings: Settings,
    *,
    user_prompt: str,
    system_prompt: str,
    max_rounds: int | None = None,
) -> str:
    """One user turn with Toolbox tools (happy path)."""
    cap = max_rounds if max_rounds is not None else settings.agent_max_tool_rounds
    oai = openrouter_client(settings)
    messages = initial_messages(system_prompt, user_prompt, cap)

    # Keep the client open for the whole user turn so every tool call shares one MCP session.
    with ToolboxSyncClient(
        settings.toolbox_base_url,
        protocol=Protocol.MCP_LATEST,
    ) as tb:
        loaded = tb.load_toolset("combined")  # name from toolbox/config/tools/04_toolsets.yaml
        by_name = tool_by_name(loaded)
        oai_tools = toolbox_tools_to_openai(loaded)

        for _ in range(cap):
            resp = call_model(oai, settings, messages, oai_tools)
            msg = resp.choices[0].message
            if msg.tool_calls:
                # Record what the model asked for, run tools, then ask again with a longer transcript.
                append_assistant_with_tool_calls(messages, msg)
                append_tool_outputs(messages, msg, by_name)
                continue
            # No ``tool_calls``: treat assistant text as the final answer for this user message.
            return (msg.content or "").strip()

    # Only runs if ``cap`` rounds all ended with tool calls (never a text-only finish inside the loop).
    return "Round limit reached without a final text reply."


## 4. Example A — structured data then bios

**Cross-database flow (Postgres first, then Mongo):** The question is anchored in **facts**
that live mainly in **Postgres** (editions, NOCs, medal events, athlete ids). A sensible agent
path is: use **`pg_*`** tools to narrow a Games year and country, list medalists, and collect
stable **athlete ids** from the relational results.

Only after those ids exist should the model lean on **Mongo** **`mongo_*`** tools for
**biography text** (longer narrative fields that are not tidy columns). That is the tutorial
pattern **structured data first, bios second**—one database supplies the spine, the other adds
story and quotes.

The next cell defines the system and user prompts,
then runs `chat_with_tools` so you can watch the model interleave `pg_*` and `mongo_*` calls.

**Docs:** [MCP Toolbox introduction](https://mcp-toolbox.dev/documentation/introduction/); [OpenRouter](https://openrouter.ai/docs/quickstart); [function calling](https://platform.openai.com/docs/guides/function-calling).


In [ ]:
TUTORIAL_SYSTEM_PROMPT = (
    "You are a tutorial assistant with Olympic data tools (relational + document). "
    "Use tools for facts; do not invent athlete, edition, or country codes. "
    "Call tools yourself—do not ask the user to run queries. "
    "Keep tool use reasonable in depth (a few biography lookups per answer is enough). "
    "Prefer concise answers with citations to ids returned by tools."
)

PROMPT_A = (
    "Choose a Summer Olympic Games held between 1992 and 2020, and choose one of these "
    "countries to focus on: United States, Australia, Japan, France, or Kenya. Who won "
    "medals for that country at those Games (any sport)? Use the data tools, mention a "
    "few medalists with a line or two from their biographies when you can, and cite "
    "athlete ids from the tools."
)

answer_a = chat_with_tools(
    settings,
    user_prompt=PROMPT_A,
    system_prompt=TUTORIAL_SYSTEM_PROMPT,
)
print(answer_a)

## 5. Example B — biography search then cross-check

**Cross-database flow (Mongo first, then Postgres):** The hook is a **sport theme** in natural
language, which maps cleanly to **Mongo** **`mongo_*`** text search over biographies. The agent
should surface one or a few plausible athletes from that **document** side first.

The teaching goal is then **verification in Postgres**: use **`pg_*`** tools to check medals,
starts, and editions for those athletes so the final answer compares **what the bio emphasizes**
with **what the relational tables record**.

This notebook picks **one random sport phrase** from a short list and omits an extra "which search hit" nudge that tighter demos sometimes add.

The next cell builds the user prompt and runs `chat_with_tools`.

**Docs:** same as example A: [Toolbox](https://mcp-toolbox.dev/documentation/introduction/), [OpenRouter](https://openrouter.ai/docs/quickstart), [function calling](https://platform.openai.com/docs/guides/function-calling).


In [ ]:
import random

TUTORIAL_SYSTEM_PROMPT = (
    "You are a tutorial assistant with Olympic data tools (relational + document). "
    "Use tools for facts; do not invent athlete, edition, or country codes. "
    "Call tools yourself—do not ask the user to run queries. "
    "Keep tool use reasonable in depth (a few biography lookups per answer is enough). "
    "Prefer concise answers with citations to ids returned by tools."
)

NOTEBOOK_B_SPORTS = (
    "decathlon",
    "pole vault",
    "100 metres freestyle",
    "balance beam",
    "badminton",
)


def notebook_prompt_b() -> tuple[str, str]:
    sport = random.choice(NOTEBOOK_B_SPORTS)
    user = (
        f"Find athlete biography material tied to Olympic sport or event, using the theme "
        f"{sport!r} in a text-oriented search over the biography data. "
        f"Then cross-check that person against structured Olympic tables (starts, medals, editions) "
        f"and summarize: what the bio emphasizes versus what the relational data supports. "
        f"Cite ids where helpful."
    )
    return user, sport


prompt_b, sport_b = notebook_prompt_b()
print("Sport theme:", sport_b)

answer_b = chat_with_tools(
    settings,
    user_prompt=prompt_b,
    system_prompt=TUTORIAL_SYSTEM_PROMPT,
)
print(answer_b)

# PART II SECTION 1 - Try calling MCP!

## Problem 1 — Athletes who did not compete for their birth country

**Postgres-only flow:** Both pieces of information you need — an athlete's `birth_country` and the `country_represented` in their events — live in the Postgres tables. A sensible agent path is: pick a country and a Games year, use `pg_medal_events_for_country` to get the medalists for that team, then look up each athlete's `birth_country` via `pg_search_athletes_by_name` and flag anyone whose birth country differs from the country they competed for.

The teaching goal is **chaining two tools together**: the first call returns a list of athletes, and the second call is used to enrich each one with a field the first tool did not return. This is a common pattern when no single tool answers the full question.

This problem is anchored to the **2012 London Summer Games** — write a plain string prompt, no f-strings needed.

The next cell holds your prompt and runs `chat_with_tools`.

**Docs:** same as the examples above: [Toolbox](https://mcp-toolbox.dev/documentation/introduction/), [OpenRouter](https://openrouter.ai/docs/quickstart), [function calling](https://platform.openai.com/docs/guides/function-calling).

In [ ]:
HOMEWORK_1_SYSTEM_PROMPT = (
    "You are a tutorial assistant with Olympic data tools (relational + document). "
    "Use tools for facts; do not invent athlete, edition, or country codes. "
    "Call tools yourself—do not ask the user to run queries. "
    "When checking birth countries, look up each athlete by name and compare birth_country "
    "to the country they competed for. "
    "Prefer concise answers with citations to ids returned by tools."
)

# TODO: Write a prompt asking about athletes at the 2012 Summer Olympics who competed
# for a country different from their birth country.
# DEV TODO: REMOVE THIS LINE AND ANSWER KEY BELOW
prompt_1 = (
    "For the 2012 Summer Olympics, pick one country that won at least a few medals. "
    "Get the list of medalists for that country, then look up each athlete by name to find "
    "their birth country. List any athletes whose birth country differs from the country "
    "they competed for, and note the difference. Cite athlete_ids in your answer."
)

answer_1 = chat_with_tools(
    settings,
    user_prompt=prompt_1,
    system_prompt=HOMEWORK_1_SYSTEM_PROMPT,
)
print(answer_1)


## Problem 2 — Highest gold medal winner in a random country

**Cross-database flow (Postgres first, then Mongo):** The question starts in **structured data**—you need counts of gold medals per athlete, which lives in Postgres. A sensible agent path is: resolve the country name to a NOC code, then call `pg_medal_events_for_country` across one or more editions to collect gold-medal rows, identify which athlete appears most often, and only then pull their biography from Mongo.

The teaching goal is **aggregation via the agent**: the existing tools return individual medal rows rather than pre-counted tallies, so the model must group and rank the results itself before reaching for the biography.

This cell picks **one random country** from a short list and builds a prompt that asks the agent to work through the full flow.

The next cell builds the user prompt and runs `chat_with_tools`.

**Docs:** same as the examples above: [Toolbox](https://mcp-toolbox.dev/documentation/introduction/), [OpenRouter](https://openrouter.ai/docs/quickstart), [function calling](https://platform.openai.com/docs/guides/function-calling).

In [ ]:
import random

HOMEWORK_2_SYSTEM_PROMPT = (
    "You are a tutorial assistant with Olympic data tools (relational + document). "
    "Use tools for facts; do not invent athlete, edition, or country codes. "
    "Call tools yourself—do not ask the user to run queries. "
    "When counting medals, tally the rows the tools return and rank them yourself. "
    "Prefer concise answers with citations to ids returned by tools."
)

HOMEWORK_2_COUNTRIES = (
    # TODO: Add in a few countries to get started! For example: "United States", "Kenya"
    # DEV TODO: REMOVE THIS LINE AND ANSWER KEY BELOW
    "United States",
    "Kenya",
    "Australia",
    "Cuba",
    "Hungary",
    "Norway",
)


def homework_prompt_2() -> tuple[str, str]:
    country = random.choice(HOMEWORK_2_COUNTRIES)
    # TODO: Write your prompt here! Use an f-string to insert the country name. Example: f"I live in {country}"
    # DEV TODO: REMOVE THIS LINE AND ANSWER KEY BELOW
    user = (
        f"For the country {country!r}, find the athlete who has won the most gold medals "
        f"across all Summer Olympic Games in the data. "
        f"Use the structured tools to resolve the country's NOC code, collect gold-medal rows, "
        f"and count how many gold medals each athlete won—then identify the top winner. "
        f"Finally, fetch their biography from the document database and include a sentence or two "
        f"from it. Cite the athlete_id and gold medal count in your answer."
    )
    return user, country


prompt_2, country_2 = homework_prompt_2()
print("Country:", country_2)

answer_2 = chat_with_tools(
    settings,
    user_prompt=prompt_2,
    system_prompt=HOMEWORK_2_SYSTEM_PROMPT,
)
print(answer_2)


## Problem 3 — Best performing country in a sport over a range of years

**Cross-database flow (Postgres first, then Mongo):** Finding the best team over a span of Games requires looping across multiple editions. The agent should use `pg_list_games_by_year` to find every edition in the year range, call `pg_medal_events_for_country` with a sport filter for each edition to collect medal rows, tally the results across years, and then pull a Mongo biography for a standout athlete.

The teaching goal is **multi-variable prompting**: this problem has four moving parts that all need to land in the right place in your prompt — a randomly chosen country, a randomly chosen sport, and a `year_min`/`year_max` range that **you** set. Getting the agent to use all four correctly in one coherent query is the challenge.

This cell picks **one random country** and **one random sport**. You set `year_min` and `year_max` yourself, then write an f-string prompt that weaves all four variables together.

The next cell holds your variables and prompt, then runs `chat_with_tools`.

**Docs:** same as the examples above: [Toolbox](https://mcp-toolbox.dev/documentation/introduction/), [OpenRouter](https://openrouter.ai/docs/quickstart), [function calling](https://platform.openai.com/docs/guides/function-calling).

In [ ]:
import random

HOMEWORK_3_SYSTEM_PROMPT = (
    "You are a tutorial assistant with Olympic data tools (relational + document). "
    "Use tools for facts; do not invent athlete, edition, or country codes. "
    "Call tools yourself—do not ask the user to run queries. "
    "When spanning multiple Games, call medal tools once per edition and tally across them. "
    "Prefer concise answers with citations to ids returned by tools."
)

HOMEWORK_3_COUNTRIES = (
    # TODO: Add a few countries to pick from! For example: "United States", "China"
    # DEV TODO: REMOVE THIS LINE AND ANSWER KEY BELOW
    "United States",
    "China",
    "Germany",
    "Australia",
    "Jamaica",
)

HOMEWORK_3_SPORTS = (
    # TODO: Add a few Olympic sports to pick from! For example: "swimming", "athletics"
    # DEV TODO: REMOVE THIS LINE AND ANSWER KEY BELOW
    "swimming",
    "athletics",
    "gymnastics",
    "rowing",
    "cycling",
    "bobsledding"
)

# TODO: Set the year range you want to search across.
year_min = None  # replace with a year, e.g. 1992
year_max = None  # replace with a year, e.g. 2012


def homework_prompt_3() -> tuple[str, str, str]:
    country = random.choice(HOMEWORK_3_COUNTRIES)
    sport = random.choice(HOMEWORK_3_SPORTS)
    # TODO: Write your prompt here! Use an f-string to include country, sport, year_min, and year_max.
    # DEV TODO: REMOVE THIS LINE AND ANSWER KEY BELOW
    user = (
        f"For the country {country!r}, find all medal results in {sport!r} events "
        f"at Summer Olympic Games between {year_min} and {year_max}. "
        f"First look up the editions in that year range, then collect medal rows for each edition "
        f"filtered to {sport!r}, and tally the total medals across all those Games. "
        f"Identify the edition where {country!r} performed best in {sport!r}, then fetch a "
        f"biography for one of the top medalists from that edition. Cite athlete_ids."
    )
    return user, country, sport


prompt_3, country_3, sport_3 = homework_prompt_3()
print(f"Country: {country_3} | Sport: {sport_3} | Years: {year_min}-{year_max}")

answer_3 = chat_with_tools(
    settings,
    user_prompt=prompt_3,
    system_prompt=HOMEWORK_3_SYSTEM_PROMPT,
)
print(answer_3)


# PART II SECTION 2 - Make your own MCP tool!

## Problem 4 — Write the SQL for a medal tally tool

**Write your own SQL:** None of the existing tools can answer "which country topped the medal table at a given Games?" without calling `pg_medal_events_for_country` for every country separately — which is impractical. Your job is to write the SQL query that closes that gap for a new `pg_medal_tally_by_edition` tool.

**What the query should return:** One row per country with gold, silver, and bronze counts plus a total, sorted by gold descending. It takes a single parameter — `$1` — which will be the `edition_id` (an integer you can get from `pg_list_games_by_year`).

**The key SQL pattern:** `COUNT(*) FILTER (WHERE ...)` lets you pivot medal types into columns in a single aggregation pass — no subqueries needed.

The next cell has the prompt and system prompt already written. Your only job is to fill in `medal_tally_sql` with a working query.

**Docs:** PostgreSQL [`COUNT` with `FILTER`](https://www.postgresql.org/docs/current/sql-expressions.html#SYNTAX-AGGREGATES); schema reference: `athlete_events(edition_id, country_represented, medal)` joined to `olympic_games(edition_id)`.

In [ ]:
# TODO: Write the SQL for the pg_medal_tally_by_edition tool.
# Requirements:
#   - Filter to edition_id = $1 (the single parameter this tool will accept)
#   - Count gold, silver, and bronze medals separately per country
#   - Include a total medal count column
#   - Group by country_represented, sort by gold DESC
# Hint: use COUNT(*) FILTER (WHERE ae.medal = 'Gold') for each medal type
# DEV TODO: REMOVE THIS LINE AND ANSWER KEY BELOW
medal_tally_sql = """
SELECT
    ae.country_represented                                              AS noc,
    COUNT(*) FILTER (WHERE ae.medal = 'Gold')                         AS gold,
    COUNT(*) FILTER (WHERE ae.medal = 'Silver')                       AS silver,
    COUNT(*) FILTER (WHERE ae.medal = 'Bronze')                       AS bronze,
    COUNT(*) FILTER (WHERE ae.medal IS NOT NULL
                      AND TRIM(ae.medal) <> '')                       AS total
FROM athlete_events ae
WHERE ae.edition_id = $1
  AND ae.medal IS NOT NULL
  AND TRIM(ae.medal) <> ''
GROUP BY ae.country_represented
ORDER BY gold DESC, silver DESC, bronze DESC
LIMIT 50
"""

print(medal_tally_sql)

# Once your SQL is added to the toolset and Toolbox is restarted, run this to test it.
HOMEWORK_4_SYSTEM_PROMPT = (
    "You are a tutorial assistant with Olympic data tools (relational + document). "
    "Use tools for facts; do not invent athlete, edition, or country codes. "
    "Call tools yourself—do not ask the user to run queries. "
    "Use pg_list_games_by_year to resolve a year to an edition_id, then use "
    "pg_medal_tally_by_edition to get the full medal table for that edition. "
    "Prefer concise answers with citations to ids returned by tools."
)

prompt_4 = (
    "Use the available tools to find the edition_id for the 2008 Summer Olympics, "
    "then call pg_medal_tally_by_edition with that edition_id to get the full medal table. "
    "Which country finished first, second, and third in gold medals? "
    "Cite the edition_id and the NOC codes from the tool output."
)

answer_4 = chat_with_tools(
    settings,
    user_prompt=prompt_4,
    system_prompt=HOMEWORK_4_SYSTEM_PROMPT,
)
print(answer_4)
